In [25]:
from dataclasses import dataclass


@dataclass
class ModelTrainerConfig:
    root_dir: str
    training_data_path: str
    trained_model_dir: str
    train_data_path: str
    test_data_path: str
    metrics_file_path: str
    test_size: float
    random_state: int
    max_training_rows: int
    target_column: str
    datetime_column: str
    model_params: dict  

In [26]:
from dataclasses import dataclass


@dataclass
class ModelTrainerArtifact:
    trained_model_dir: str
    train_data_path: str
    test_data_path: str
    model_metrics_path: str

In [27]:
from src.utils.main_utils import read_yaml, create_directories
from src.constants import CONFIG_FILE_PATH, SCHEMA_FILE_PATH
from pathlib import Path
import os



class ConfigurationManager:
    
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

        create_directories([
            self.config.artifacts_root,
            self.config.data_validation.root_dir,
            self.config.model_trainer.root_dir,
            self.config.model_trainer.trained_model_dir,
        ])

    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config = self.config.model_trainer

        create_directories([
            config.root_dir,
            config.trained_model_dir,
        ])

        return ModelTrainerConfig(
            root_dir=config.root_dir,
            training_data_path=self.config.feature_engineering.final_feature_path,
            trained_model_dir=config.trained_model_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            metrics_file_path=config.metrics_file_path,
            test_size=config.test_size,
            random_state=config.random_state,
            max_training_rows=config.max_training_rows,
            target_column=config.target_column,
            datetime_column=self.config.feature_engineering.datetime_column,
            model_params=config.model_params,
        )

In [28]:
import json
import os
import sys
from importlib import import_module

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from src.logger import logging
from src.exception import MyException

try:
	XGBClassifier = import_module("xgboost").XGBClassifier
except Exception:
	XGBClassifier = None


class ModelTrainer:
	def __init__(self, config: ModelTrainerConfig):
		self.config = config

	def _load_data(self) -> pd.DataFrame:
		if not os.path.exists(self.config.training_data_path):
			raise FileNotFoundError(f"Training data not found at {self.config.training_data_path}")

		df = pd.read_csv(self.config.training_data_path)
		if self.config.datetime_column in df.columns:
			df[self.config.datetime_column] = pd.to_datetime(df[self.config.datetime_column])
		return df

	def _resolve_target_column(self, df: pd.DataFrame) -> str:
		if self.config.target_column in df.columns:
			return self.config.target_column

		fallback_columns = [
			column
			for column in df.columns
			if column.startswith("failure_within_next_") or column.startswith("target_failure_")
		]
		if fallback_columns:
			logging.warning(
				f"Configured target '{self.config.target_column}' not found. Using '{fallback_columns[0]}' instead."
			)
			return fallback_columns[0]

		raise KeyError(f"Target column '{self.config.target_column}' not found in training data")

	def _chronological_split(self, df: pd.DataFrame, target_column: str):
		working_df = df.copy()

		if self.config.datetime_column in working_df.columns:
			working_df = working_df.sort_values(self.config.datetime_column)
		else:
			working_df = working_df.sort_index()

		if self.config.max_training_rows and len(working_df) > self.config.max_training_rows:
			logging.info(
				f"Limiting training data to the first {self.config.max_training_rows} chronological rows "
				f"out of {len(working_df)} total rows for the dry-run."
			)
			working_df = working_df.iloc[: self.config.max_training_rows].copy()

		working_df = working_df.reset_index(drop=True)
		split_index = int(len(working_df) * (1 - self.config.test_size))
		split_index = max(1, min(split_index, len(working_df) - 1))

		train_df = working_df.iloc[:split_index].copy().reset_index(drop=True)
		test_df = working_df.iloc[split_index:].copy().reset_index(drop=True)

		train_y = train_df[target_column].astype(int)
		test_y = test_df[target_column].astype(int)

		train_x = train_df.drop(columns=[target_column])
		test_x = test_df.drop(columns=[target_column])

		drop_columns = [self.config.datetime_column, "machineID"]
		train_x = train_x.drop(columns=[column for column in drop_columns if column in train_x.columns])
		test_x = test_x.drop(columns=[column for column in drop_columns if column in test_x.columns])

		train_x = pd.get_dummies(train_x, drop_first=False)
		test_x = pd.get_dummies(test_x, drop_first=False)
		test_x = test_x.reindex(columns=train_x.columns, fill_value=0)
		train_x = train_x.loc[:, ~train_x.columns.duplicated()].copy()
		test_x = test_x.loc[:, ~test_x.columns.duplicated()].copy()

		train_df.to_csv(self.config.train_data_path, index=False)
		test_df.to_csv(self.config.test_data_path, index=False)

		return train_x, test_x, train_y, test_y

	def _build_candidate_models(self):
		model_params = self.config.model_params or {}

		models = {
			"logistic_regression": LogisticRegression(
				**model_params.get("logistic_regression", {}),
			),
			"random_forest": RandomForestClassifier(
				random_state=self.config.random_state,
				**model_params.get("random_forest", {}),
			),
		}

		if XGBClassifier is not None:
			xgb_params = dict(model_params.get("xgboost", {}))
			xgb_params.setdefault("random_state", self.config.random_state)
			xgb_params.setdefault("objective", "binary:logistic")
			xgb_params.setdefault("eval_metric", "logloss")
			models["xgboost"] = XGBClassifier(**xgb_params)
		else:
			logging.warning("xgboost is not available in the current environment; skipping the XGBoost candidate.")

		return models

	def _evaluate_model(self, y_true, y_pred, y_probability=None):
		metrics = {
			"accuracy": accuracy_score(y_true, y_pred),
			"precision": precision_score(y_true, y_pred, zero_division=0),
			"recall": recall_score(y_true, y_pred, zero_division=0),
			"f1": f1_score(y_true, y_pred, zero_division=0),
		}

		if y_probability is not None:
			try:
				metrics["roc_auc"] = roc_auc_score(y_true, y_probability)
			except Exception:
				metrics["roc_auc"] = None
		else:
			metrics["roc_auc"] = None

		return metrics

	def initiate_model_trainer(self):
		try:
			logging.info("Starting model training...")

			os.makedirs(self.config.root_dir, exist_ok=True)
			os.makedirs(self.config.trained_model_dir, exist_ok=True)

			df = self._load_data()
			target_column = self._resolve_target_column(df)
			train_x, test_x, train_y, test_y = self._chronological_split(df, target_column)

			candidate_models = self._build_candidate_models()
			metrics_report = {}
			best_model_name = None
			best_model_score = float("-inf")

			for model_name, model in candidate_models.items():
				logging.info(f"Training candidate model: {model_name}")
				model.fit(train_x, train_y)

				predictions = model.predict(test_x)
				probabilities = None
				if hasattr(model, "predict_proba"):
					probabilities = model.predict_proba(test_x)[:, 1]

				metrics = self._evaluate_model(test_y, predictions, probabilities)
				metrics_report[model_name] = metrics

				model_path = os.path.join(self.config.trained_model_dir, f"{model_name}.joblib")
				joblib.dump(model, model_path)
				metrics_report[model_name]["model_path"] = model_path

				candidate_score = metrics["f1"] if metrics["f1"] is not None else float("-inf")
				if candidate_score > best_model_score:
					best_model_score = candidate_score
					best_model_name = model_name

			metrics_payload = {
				"target_column": target_column,
				"best_model": best_model_name,
				"models": metrics_report,
			}

			with open(self.config.metrics_file_path, "w", encoding="utf-8") as metrics_file:
				json.dump(metrics_payload, metrics_file, indent=4)

			logging.info("Model training completed successfully.")

			return ModelTrainerArtifact(
				trained_model_dir=self.config.trained_model_dir,
				train_data_path=self.config.train_data_path,
				test_data_path=self.config.test_data_path,
				model_metrics_path=self.config.metrics_file_path,
			)

		except Exception as e:
			logging.exception(f"Model training failed: {e}")
			raise MyException(e, sys)


In [30]:
%pwd

'c:\\Users\\dipjy\\OneDrive\\Desktop\\Career_2026\\Projects\\Mlops_projects\\Predictive_maintanance_system'

In [23]:
cd ..

c:\Users\dipjy\OneDrive\Desktop\Career_2026\Projects\Mlops_projects\Predictive_maintanance_system


In [31]:

from src.logger import logging
from src.exception import MyException
import sys


class TrainingPipeline:
    def __init__(self):
        self.config_manager = ConfigurationManager()

    def start_model_trainer(self):
        """
        Build the ModelTrainer component and run the model training flow.
        """
        model_trainer_config = self.config_manager.get_model_trainer_config()
        model_trainer = ModelTrainer(config=model_trainer_config)
        return model_trainer.initiate_model_trainer()


STAGE_NAME_4 = "Model Training Stage"

if __name__ == "__main__":
    try:
        
        pipeline = TrainingPipeline()
        logging.info(f">>>>>> {STAGE_NAME_4} started <<<<<<")
        model_trainer_artifact = pipeline.start_model_trainer()
        logging.info(f">>>>>> {STAGE_NAME_4} completed successfully <<<<<<\n\nx==========x")
    except Exception as e:
        logging.exception(e)
        raise MyException(e, sys)

[ 2026-05-08 14:28:13,270 ] root - INFO - yaml file: config\config.yaml loaded successfully
[ 2026-05-08 14:28:13,292 ] root - INFO - yaml file: config\schema.yaml loaded successfully
[ 2026-05-08 14:28:13,302 ] root - INFO - created directory at: artifacts
[ 2026-05-08 14:28:13,307 ] root - INFO - created directory at: artifacts/data_validation
[ 2026-05-08 14:28:13,307 ] root - INFO - created directory at: artifacts/model_trainer
[ 2026-05-08 14:28:13,324 ] root - INFO - created directory at: artifacts/model_trainer/models
[ 2026-05-08 14:28:13,329 ] root - INFO - >>>>>> Model Training Stage started <<<<<<
[ 2026-05-08 14:28:13,335 ] root - INFO - created directory at: artifacts/model_trainer
[ 2026-05-08 14:28:13,342 ] root - INFO - created directory at: artifacts/model_trainer/models
[ 2026-05-08 14:28:13,348 ] root - INFO - Starting model training...


[ 2026-05-08 14:29:29,954 ] root - INFO - Limiting training data to the first 50000 chronological rows out of 784181 total rows for the dry-run.
[ 2026-05-08 14:29:45,733 ] root - INFO - Training candidate model: logistic_regression
[ 2026-05-08 14:30:11,060 ] root - INFO - Training candidate model: random_forest
[ 2026-05-08 14:30:17,733 ] root - INFO - Training candidate model: xgboost
[ 2026-05-08 14:30:19,763 ] root - INFO - Model training completed successfully.
[ 2026-05-08 14:30:19,918 ] root - INFO - >>>>>> Model Training Stage completed successfully <<<<<<

x==========x
